IA & Data science (LU3IN0226) -- 2024-2025
--------
*&copy; Equipe pédagogique: Christophe Marsala, Olivier Schwander, Jean-Noël Vittaut.*


# TD-TME 6 : Apprentissage pour le texte

<font size="+1" color="RED"><b>[Q]</b></font> **Indiquer dans la boîte ci-dessous vos noms et prénoms :**

Ines Harraoui
Yanis Saadi Dit Saada

<font color="RED" size="+1"><b>[Q]</b></font> **Renommer ce notebook**

Tout en haut de cette page, cliquer sur <tt>tme-06</tt> et rajouter à la suite de <tt>tme-06</tt> les noms des membres du binômes séparés par un tiret.

<font color="RED" size="+1">IMPORTANT: soumission de votre fichier final</font>

**Nom à donner au fichier à poster** : *tme-06-Nom1_Nom2.ipynb* 
- *Nom1* et *Nom2* : noms des membres du binôme
- ne pas compresser ou faire une archive: il faut rendre le fichier ipython tel quel, éventuellement, si vous avez d'autres fichiers vous les rendez séparément.

**Echancier pour la soumission de votre compte-rendu:**
- le compte-rendu d'une séance doit être remis obligatoirement <font color="RED">avant la séance suivante</font>.

**Le compte-rendu est soumis sur la page Moodle.**

In [49]:
# - - - - - - - - - - - - - - - - - -
# imports utiles
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mtpl
%matplotlib inline  

import math
import time
import sys

# Les instructions suivantes sont utiles pour recharger automatiquement 
# le code modifié dans les librairies externes
%load_ext autoreload
%autoreload 2

# - - - - - - - - - - - - - - - - - -
# Information sur l'environnent utilisé ici:
print("Version python et des librairies:")
print("\tPython ",sys.version)
print("\tpandas: ",pd.__version__)
print("\tnumpy: ",np.__version__)
print("\tmatplotlib: ",mtpl.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Version python et des librairies:
	Python  3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
	pandas:  2.0.3
	numpy:  1.24.3
	matplotlib:  3.7.2


In [50]:
# Importation de votre librairie iads:
# La ligne suivante permet de préciser le chemin d'accès à la librairie iads
import sys
sys.path.append('../')   # iads doit être dans le répertoire père du répertoire courant !

# Importation de la librairie iads
import iads as iads

# importation de Classifiers
from iads import Classifiers as classif

# importation de utils
from iads import utils as ut

# importation de evaluation
from iads import evaluation as ev



# Objectifs de ce TME

<div class="alert alert-block alert-warning">
Ce TME a pour but d'implémenter des fonctions pour traiter un corpus textuel et un nouvel algorithme d'apprentissage vu dans le cours 6. 

Pour expérimenter, on utilise la base `SMS spam Collection` qui contient 5572 messages associés à 2 labels: `spam` et `ham`. 
</div>

In [51]:
# Chargement du dataset

df_spam = pd.read_csv('data/spam.csv', sep='\t', encoding = 'latin1')
df_spam

,label,message
0,ham,Go until jurong point crazy.. Available only ...
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,Nah I don't think he goes to usf he lives aro...
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,Pity * was in mood for that. So...any other s...
5570,ham,The guy did some bitching but I acted like i'd...


In [52]:
# Valeurs du label:
df_spam['label'].unique()

array(['ham', 'spam'], dtype=object)

In [53]:
# On met les labels dans une liste car cela sera utile:
les_labels = df_spam['label'].unique()

<font color="RED" size="+1"><b>[Q]</b></font> En utilisant `value_counts` (voir la doc de la librairie pandas), afficher le nombre d'exemples de chaque classe dans la base.

In [54]:
df_spam['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

## Prétraitements

### Nettoyage des données

Pour pouvoir travailler sur les messages, on commence par les nettoyer : il faut enlever les caractères de ponctuations, les articles, les mots trop courants, etc. (voir le cours 6).

Pour traiter les données textuelles (colonne `message` du dataset), on utilise les fonctions de la librairie `string`.

In [55]:
import string

print("Caractères de ponctuations : ", string.punctuation)

print("Mise en minuscules (pour homogénéiser l'écriture : ", "May the Force be with you!".lower())

print("Découper une phrase avec espace (retourne une liste):", "LU3IN026 est l'UE d'IA et Sciences des données.".split())

print("Découper une phrase avec apostrophe (retourne une liste):", "LU3IN026 est l'UE d'IA et Sciences des données.".split("'"))



Caractères de ponctuations :  !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Mise en minuscules (pour homogénéiser l'écriture :  may the force be with you!
Découper une phrase avec espace (retourne une liste): ['LU3IN026', 'est', "l'UE", "d'IA", 'et', 'Sciences', 'des', 'données.']
Découper une phrase avec apostrophe (retourne une liste): ['LU3IN026 est l', 'UE d', 'IA et Sciences des données.']


In [56]:
# Avec le dataframe du dataset:
print("Premier message du train: ",df_spam['message'].iloc[0])

print("Résultat d'un découpage: ",df_spam['message'].iloc[0].split())

Premier message du train:  Go until jurong point  crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Résultat d'un découpage:  ['Go', 'until', 'jurong', 'point', 'crazy..', 'Available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet...', 'Cine', 'there', 'got', 'amore', 'wat...']


In [57]:
# Certains éléments d'une phrase ne sont pas utiles pour le traitement, par exemple en anglais:
mots_inutiles = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should', "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `nettoyage` qui prend une chaîne de caractères et rend la chaîne après nettoyage: 1) mise en minuscules 2) remplacement des caractères de ponctuation par un espace (SAUF l'apostrophe qui doit rester car elle va être utilisée pour les mots inutiles).


In [58]:
def nettoyage(texte):
    texte = texte.lower()
    
    texte_nettoye = ""
    
    for c in texte:
        if c not in string.punctuation.replace("'", ""):
            texte_nettoye += c
        else:
            texte_nettoye += " "  
    
    return texte_nettoye    


In [59]:
nettoyage("LU3IN026 est l'UE d'IA et Sciences des données.")

"lu3in026 est l'ue d'ia et sciences des données "

In [60]:
nettoyage(df_spam['message'].iloc[0])

'go until jurong point  crazy   available only in bugis n great world la e buffet    cine there got amore wat   '

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `text2vect` qui prend une chaîne de caractères ainsi qu'une liste de mots inutiles et rend la liste composée par les mots de cette chaîne obtenus, après son nettoyage et après avoir enlevé les mots inutiles.



In [61]:
def text2vect(texte, mots_inutiles):

    texte_nettoye = nettoyage(texte)
    
    mots = texte_nettoye.split()
    
    mots_filtrés = [mot for mot in mots if mot not in mots_inutiles]
    
    return mots_filtrés



In [62]:
text2vect("May the Force be with you!",mots_inutiles)

['may', 'force']

In [63]:
text2vect("You shan't pass!",mots_inutiles)

['pass']

In [64]:
text2vect(df_spam['message'].iloc[0],mots_inutiles)

['go',
 'jurong',
 'point',
 'crazy',
 'available',
 'bugis',
 'n',
 'great',
 'world',
 'la',
 'e',
 'buffet',
 'cine',
 'got',
 'amore',
 'wat']

<font color="RED" size="+1"><b>[Q]</b></font> Ajouter une nouvelle colonne de nom `les_mots` au dataframe `df_spam` pour laquelle chaque ligne contient le résultat de l'application de `text2vect` sur le message de l'exemple correspondant.


In [65]:
# A COMPLETER 
les_mots_list = []

for message in df_spam['message']:
    les_mots_list.append(text2vect(message, mots_inutiles))

df_spam['les_mots'] = les_mots_list

# ----------------------------------------------------------

df_spam

,label,message,les_mots
0,ham,Go until jurong point crazy.. Available only ...,"[go, jurong, point, crazy, available, bugis, n..."
1,ham,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,ham,U dun say so early hor... U c already then say...,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,Nah I don't think he goes to usf he lives aro...,"[nah, think, goes, usf, lives, around, though]"
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"[2nd, time, tried, 2, contact, u, u, å£750, po..."
5568,ham,Will Ì_ b going to esplanade fr home?,"[ì, b, going, esplanade, fr, home]"
5569,ham,Pity * was in mood for that. So...any other s...,"[pity, mood, suggestions]"
5570,ham,The guy did some bitching but I acted like i'd...,"[guy, bitching, acted, like, i'd, interested, ..."


### Découpage du dataset en 2 ensembles train et test

<font color="RED" size="+1"><b>[Q]</b></font> Pour mettre au point nos fonctions dans ce TME, on va travailler sur une partie du dataset, par exemple, on peut prendre 5% des exemples (en respectant la distribution des classes, donc en prenant 5% d'exemples de chaque label).
Compléter le code ci-dessous pour définir les 2 variables `df_train` et `df_test` qui contiendront chacune 1 dataframe, partie de `df_spam`. Pour construire `df_train` on prend aléatoirement 5% des exemples de chaque classe, comme on l'a déjà fait dans un TME précédent (cf. `np.random.shuffle`).


In [66]:
np.random.seed(42)

# pourcentage d'exemples de chaque classe à garder:
taux = 0.05    # ici on prend 5% 

# déclaration des variables qui seront initialisées dans la boucle:
df_train = None  
df_test = None
for l in les_labels:
    nb_total = df_spam['label'].value_counts()[l]
    nb_pris = int(nb_total*taux) 
    print(f"Nombre d'exemples du label {l} pris pour apprendre: {nb_pris}")

    les_ids = df_spam[df_spam['label']==l].index.to_list()
    np.random.shuffle(les_ids)

    # ################################# COMPLETER ICI 
    train_ids = les_ids[:nb_pris]  
    test_ids = les_ids[nb_pris:] 

    # Créer les DataFrames df_train et df_test pour chaque label
    if df_train is None:
        df_train = df_spam.loc[train_ids]  # Initialiser df_train avec les premiers indices
    else:
        df_train = pd.concat([df_train, df_spam.loc[train_ids]], axis=0)  # Ajouter les nouvelles données

    if df_test is None:
        df_test = df_spam.loc[test_ids]   # Initialiser df_test avec les indices restants
    else:
        df_test = pd.concat([df_test, df_spam.loc[test_ids]], axis=0)  # Ajouter les nouvelles données

    
    # ###########################################################
    
# Résultat:
print(f"Dimension de df_train:\t{df_train.shape}")
print(f"Dimension de df_test:\t{df_test.shape}")

Nombre d'exemples du label ham pris pour apprendre: 241
Nombre d'exemples du label spam pris pour apprendre: 37
Dimension de df_train:	(278, 3)
Dimension de df_test:	(5294, 3)


In [67]:
df_train

,label,message,les_mots
3714,ham,I am late so call you tomorrow morning.take ca...,"[late, call, tomorrow, morning, take, care, sw..."
1311,ham,U r too much close to my heart. If u go away i...,"[u, r, much, close, heart, u, go, away, shatte..."
548,ham,Wait &lt;#&gt; min..,"[wait, lt, gt, min]"
1324,ham,Can you call me plz. Your number shows out of ...,"[call, plz, number, shows, coveragd, area, urg..."
3184,ham,MAYBE IF YOU WOKE UP BEFORE FUCKING 3 THIS WOU...,"[maybe, woke, fucking, 3, problem]"
...,...,...,...
1852,spam,This is the 2nd time we have tried 2 contact u...,"[2nd, time, tried, 2, contact, u, u, 750, poun..."
2582,spam,3 FREE TAROT TEXTS! Find out about your love l...,"[3, free, tarot, texts, find, love, life, try,..."
3418,spam,Do you want a new Video phone? 600 anytime any...,"[want, new, video, phone, 600, anytime, networ..."
3422,spam,Had your mobile 10 mths? Update to latest Oran...,"[mobile, 10, mths, update, latest, orange, cam..."


## Calculs de probabilités

À partir d'ici on va travailler sur les données d'apprentissage (`df_train`) pour déterminer les probabilités qui serviront à classer les messages.

<font color="RED" size="+1"><b>[Q]</b></font> Construire la liste de tous les mots présents dans tous les vecteurs de `df_train`, chaque mots doit n'apparaître qu'une seule fois dans la liste obtenue. Cette liste sera stockée une fois triée dans la variable `index_mots` (et on l'appelle par la suite "index de mots").


In [68]:
index_mots = sorted(set([mot for message in df_train['message'] for mot in text2vect(message, mots_inutiles)]))

######## A COMPLETER ######


In [69]:

print("Nombre de mots trouvés: ", len(index_mots))
print("Les 10 premiers :", index_mots[0:10]) 

# pour contrôler:
for i in range(30,len(index_mots),100):
    print("\ten position ",i," --> ", index_mots[i])


Nombre de mots trouvés:  1377
Les 10 premiers : ["'", "'t", '0', '00', '01223585334', '0207', '03', '07801543489', '0800', '08000930705']
	en position  30  -->  10p
	en position  130  -->  al
	en position  230  -->  brain
	en position  330  -->  considering
	en position  430  -->  emerging
	en position  530  -->  go
	en position  630  -->  interesting
	en position  730  -->  lucky
	en position  830  -->  next
	en position  930  -->  porn
	en position  1030  -->  screaming
	en position  1130  -->  sub
	en position  1230  -->  tv
	en position  1330  -->  workin


Chaque message va maintenant être représenté comme un vecteur de valeurs 0 ou 1: ce vecteur possède autant de colonnes qu'il y a de mots dans `index_mots`. Ce vecteur est construit ainsi: pour un exemple $i$, la valeur de la colonne $j$ du vecteur vaudra 1 si la liste de mots de l'exemple $i$ contient le mot en position $j$ dans `index_mots`, ou 0 dans le cas contraire.

*Remarque:* même si un mot apparaît plusieurs fois dans la liste, cela ne compte que pour 1.

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `df2array` qui prend un dataframe `df` contenant la colonne `les_mots` ainsi qu'un index de mots et rend le `np.array` correspondant aux vecteurs de valeurs représentant les exemples de `df`. Les mots de `les_mots` qui ne sont pas dans l'index de mots ne sont pas pris en compte.  


In [70]:
def df2array(df, index_mots):
    return np.array([[1 if mot in index_mots else 0 for mot in index_mots] for mots in df['les_mots']])


In [71]:
mat_train = df2array(df_train,index_mots)

In [72]:
mat_train.shape

(278, 1377)

In [73]:
print("Message 0:", df_train['les_mots'].iloc[0])

print("\nPositions non nulles dans le vecteur d'index:")

for i in range(0, len(mat_train[0])):
    if mat_train[0][i] == 1:
        print("\tcolonne ",i," : ", index_mots[i])
    

Message 0: ['late', 'call', 'tomorrow', 'morning', 'take', 'care', 'sweet', 'dreams', 'u', 'ummifying', 'bye']

Positions non nulles dans le vecteur d'index:
	colonne  0  :  '
	colonne  1  :  't
	colonne  2  :  0
	colonne  3  :  00
	colonne  4  :  01223585334
	colonne  5  :  0207
	colonne  6  :  03
	colonne  7  :  07801543489
	colonne  8  :  0800
	colonne  9  :  08000930705
	colonne  10  :  08000938767
	colonne  11  :  08002888812
	colonne  12  :  0808
	colonne  13  :  08452810075over18's
	colonne  14  :  087104711148
	colonne  15  :  08712317606
	colonne  16  :  08712402779
	colonne  17  :  08717168528
	colonne  18  :  08717898035
	colonne  19  :  08718720201
	colonne  20  :  08718726970
	colonne  21  :  09050090044
	colonne  22  :  09061701851
	colonne  23  :  09061702893
	colonne  24  :  09064017305
	colonne  25  :  09065989182
	colonne  26  :  1
	colonne  27  :  10
	colonne  28  :  100
	colonne  29  :  10am
	colonne  30  :  10p
	colonne  31  :  11mths
	colonne  32  :  11pm
	colonne

In [74]:
# on se rappelle qu'il est possible d'extraire les vecteurs correspondant à un label donné, par exemple:
len(mat_train[df_train['label']=='ham'])

241

<font color="RED" size="+1"><b>[Q]</b></font> Construire un dictionnaire qui donne, pour chaque label $l$, pour chaque mot $m$ de l'index des mots, la fréquence de $m$ parmi les exemples de `df_train` qui ont le label $l$.


*Remarque*: penser à une solution dans boucle for sur les exemples...


In [89]:
frequences = dict()
for l in les_labels:
    frequences[l] = {mot: 0 for mot in index_mots}  
    
    for idx, row in df_train.iterrows():
        if row['label'] == l:  
            for mot in set(row['les_mots']): 
                if mot in frequences[l]: 
                    frequences[l][mot] += 1


In [90]:
# Affichage de quelques valeurs de fréquence non nulles 
print("Seuls les 10 premiers non nuls sont affichés.")        
for l in frequences:
    nb = 0
    print("Pour le label", l, ":")
    for mot in index_mots:  
        if frequences[l][mot] != 0:  
            if (nb < 10):
                print(f'\t {mot}:\t {frequences[l][mot]:0.6f}')
            nb +=1


Seuls les 10 premiers non nuls sont affichés.
Pour le label ham :
	 ':	 1.000000
	 't:	 1.000000
	 1:	 4.000000
	 100:	 1.000000
	 1st:	 1.000000
	 2:	 14.000000
	 2marrow:	 1.000000
	 2nd:	 1.000000
	 3:	 5.000000
	 30:	 1.000000
Pour le label spam :
	 0:	 1.000000
	 00:	 2.000000
	 01223585334:	 1.000000
	 0207:	 1.000000
	 03:	 1.000000
	 07801543489:	 1.000000
	 0800:	 1.000000
	 08000930705:	 2.000000
	 08000938767:	 1.000000
	 08002888812:	 1.000000


In [91]:
print("Frequences max:")
print("\tpour ham:", max(frequences['ham']), "pour le mot",index_mots[np.argmax(frequences['ham'])])
print("\tpour spam:", max(frequences['spam']), "pour le mot",index_mots[np.argmax(frequences['spam'])])


Frequences max:
	pour ham: ìï pour le mot '
	pour spam: ìï pour le mot '


## Classification de textes

<div class="alert alert-block alert-warning">
On considère deux variables $X$ et $Y$ :
    <ul>
        <li>$X$ est un message</li>
        <li>$Y$ est le label d'un message et peut prendre 2 valeurs : <code>'ham'</code> et <code>'spam'</code></li>
    </ul>

Avec les fonctions précédentes on peut représenter les messages d'un corpus de documents sous la forme d'un vecteur $X \in \{0, 1\}^p$ où $p$ est le nombre total de mots de l'index. Le $i$ème message est représenté par le vecteur ${\bf x}_i = [x_{i1} \dots x_{ip}]$, où $x_{ij}$ vaut 1 si le $j$ème mot de l'index est présent dans le message $i$, et 0 sinon.
    
Comme vu en cours, pour un classifieur bayésien, nous devons estimer, à partir de la base <code>df_train</code>, les probabilités $p(ham)$, $p(spam)$, $p(X |ham)$ et $p(X | spam)$.

Les 2 premières sont simples à calculer : on compte le nombre de fois où le label apparaît parmi les exemples.

Pour un label $Y$ et un exemple ${\bf x}$, le calcul de $p({\bf x} | Y)$ se fait en utilisant les probabilités $p(mot | Y)$ de tous les mots qui composent ${\bf x}$ (et qui sont des mots de l'index des mots). $p(mot | Y)$ est la probabilité que le mot <code>mot</code> apparaisse dans un message sachant que ce message appartient à la classe $Y$.
    
On pose ainsi

$$ p({\bf x} | Y) = \prod_{mot \in index\_mots} p(mot | Y)^{x_{mot}} \left(1 - p(mot | Y)\right)^{1 - x_{mot}} $$

où $x_{mot}$ correspond à $1$ ou $0$ selon que le `mot` apparaît dans ${\bf x}$ ou pas. Ce terme permet de retenir soit la probabilité $p(mot | Y)$ si le mot est dans ${\bf x}$, soit la probabilité qu'il n'y soit pas ($1-p(mot |Y)$).
    
Une fois que $p({\bf x} | Y)$ est calculée, on peut estimer $p(Y|X)$ grâce au théorème de Bayes (cf. cours 6):

$$p(Y|{\bf x}) = p(Y) p({\bf x} | Y)$$

avec $Y$ qui vaut soit 'ham', soit 'spam'.    

    
Une fois $p(Y{{\bf x}})$ calculée pour chaque valeur de label, pour prédire le label de ${\bf x}$ on choisit le label qui possède la plus forte probabilité.
</div>

In [ ]:
for l in ['ham', 'spam']:
    # #################### A COMPLETER 

    # p_Y = len(df_train[df_train['label'] == l]) / len(df_train)
    
    proba = len(df_train[df_train['label'] == l]) / len(df_train)  
    
    for mot in index_mots:
        if mot in df_train[df_train['label'] == l]['les_mots'].values[0]:  
            proba *= frequences[l][mot] / sum(frequences[l].values())
        else:
            proba *= 1 - (frequences[l][mot] / sum(frequences[l].values()))



<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `proba_mot` qui étant donné un mot, un label, une liste de mots, et un dictionnaire avec les fréquences des mots par label (comme `frequences` et compatible avec la liste de mots) rend $p(mot| label)$ la probabilité du mot d'appartenir au label donné.


In [ ]:
def proba_mot(mot, label, index_mots,frequences):
    if mot in index_mots:
        if label in frequences:
            if mot in frequences[label]:
                total_mots_label = sum(frequences[label].values())
                
                freq_mot = frequences[label][mot]
                
                return freq_mot/total_mots_label if total_mots_label > 0 else 0
    
    return 0

In [111]:
# probabilité d'un mot qui n'est pas dans l'index:
proba_mot("toto", "ham", index_mots, frequences)

0

In [112]:
# probabilité d'un mot de l'index pour un label qui n'existe pas:
proba_mot("visit", "cookie", index_mots, frequences)

0

In [113]:
# probabilité d'un mot de l'index pour un label qui existe:
proba_mot("call", "ham", index_mots, frequences)

0.006834910620399579

In [114]:
# probabilité d'un mot de l'index pour un label qui existe:
proba_mot("call", "spam", index_mots, frequences)

0.026825633383010434

<font color="RED" size="+1"><b>[Q]</b></font> Ecrire la fonction `proba_exemple` qui étant donné un exemple représenté sous la forme d'une liste de mots, un label, une liste de mots, et un dictionnaire comme `frequences` (compatible avec la liste de mots) rend $p(exemple|label)$ la probabilité de l'exemple d'appartenir au label.


In [118]:
def proba_exemple(exemple, label, index_mots, frequences):
    proba = 1.0 

    for mot in exemple:
        if mot in index_mots:
            p_mot_label = proba_mot(mot, label, index_mots, frequences) 
            if p_mot_label > 0:
                proba *= p_mot_label
            else:
                proba *= (1 - p_mot_label)

    return proba


In [119]:
proba_exemple(df_train['les_mots'].iloc[10], 'ham', index_mots,frequences)

1.9009882558194466e-18

In [120]:
for i in range(0,len(df_train)):
    p_spam = proba_exemple(df_train['les_mots'].iloc[i], 'spam', index_mots,frequences)
    p_ham = proba_exemple(df_train['les_mots'].iloc[i], 'ham', index_mots,frequences)
    if (p_spam>0) and (p_ham>0):
        print("Exemple",i,": p(ham)=",p_ham,"\tp(spam)=",p_spam)


Exemple 0 : p(ham)= 5.361224368771524e-30 	p(spam)= 4.4396877721516785e-10
Exemple 1 : p(ham)= 1.1753453423845264e-29 	p(spam)= 1.6550169417298755e-07
Exemple 2 : p(ham)= 4.4012945396514906e-11 	p(spam)= 0.0014903129657228018
Exemple 3 : p(ham)= 2.1367987795140955e-43 	p(spam)= 4.2875077143879104e-11
Exemple 4 : p(ham)= 4.820906216758117e-15 	p(spam)= 0.0029806259314456036
Exemple 5 : p(ham)= 9.374853552591245e-51 	p(spam)= 0.00894187779433681
Exemple 6 : p(ham)= 3.593328869899082e-79 	p(spam)= 4.411016167065751e-13
Exemple 7 : p(ham)= 2.5419832909139912e-45 	p(spam)= 8.884130943205972e-06
Exemple 8 : p(ham)= 5.528521087437983e-07 	p(spam)= 0.0014903129657228018
Exemple 9 : p(ham)= 4.837425424903324e-20 	p(spam)= 1.3326196414808958e-05
Exemple 10 : p(ham)= 1.9009882558194466e-18 	p(spam)= 2.221032735801493e-06
Exemple 11 : p(ham)= 1.1003236349128727e-11 	p(spam)= 0.0014903129657228018
Exemple 12 : p(ham)= 7.269843810942878e-117 	p(spam)= 6.041065457686111e-28
Exemple 13 : p(ham)= 1.435

<font color="RED" size="+1"><b>[Q]</b></font> À partir de `mat_train` et de `frequences` et en utilisant des opérations matricielles, on peut aussi calculer les probabilités pour chaque label de tous les exemples. En terme de temps de calcul, cela peut être plus efficace.

Donner l'instruction permettant de calculer toutes ces probabilités, puis vérifier que vous trouver les mêmes valeurs pour les exemples précédents.


In [130]:
toutes_probas = dict()
for l in les_labels:
    toutes_probas[l] =  np.exp(np.sum(mat_train * np.log(np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1) + np.sum((1 - mat_train) * np.log(1 - np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1))
    

C:\Users\inese\AppData\Local\Temp\ipykernel_20052\1254917565.py:3: RuntimeWarning: divide by zero encountered in log
  toutes_probas[l] =  np.exp(np.sum(mat_train * np.log(np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1) + np.sum((1 - mat_train) * np.log(1 - np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1))
C:\Users\inese\AppData\Local\Temp\ipykernel_20052\1254917565.py:3: RuntimeWarning: invalid value encountered in log
  toutes_probas[l] =  np.exp(np.sum(mat_train * np.log(np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1) + np.sum((1 - mat_train) * np.log(1 - np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1))
C:\Users\inese\AppData\Local\Temp\ipykernel_20052\1254917565.py:3: RuntimeWarning: invalid value encountered in multiply
  toutes_probas[l] =  np.exp(np.sum(mat_train * np.log(np.array([frequences[l].get(mot, 0) for mot in index_mots])), axis=1) + np.sum((1 - mat_train) * np.log(1 - np.array([frequ

In [127]:
for i in range(0,len(df_train)):
    p_spam = toutes_probas['spam'][i]
    p_ham = toutes_probas['ham'][i]
    if (p_spam>0) and (p_ham>0):
        print("Exemple",i,": p(ham)=",p_ham,"\tp(spam)=",p_spam)



<font color="RED" size="+1"><b>[Q]</b></font> En utilisant les fonctions écrites, calculer le taux de bonne prédiction du classifieur bayésien naïf pour chaque valeur de label pour le dataset d'apprentissage. 

*Remarque*: une solution efficace peut ne pas utiliser toutes les fonctions précédentes...

In [168]:
# La variable suivante permettra de stocker les résultats
taux = dict()
toutes_probas = dict()
for l in les_labels:
    taux[l] = dict()
    taux[l][True] = 0    # nombre de bien classés
    taux[l][False] = 0   # nombre de mal classés
    toutes_probas[l] = np.zeros(1)
tic = time.time()

# ################################## A COMPLETER

for i in range(len(df_train)):
    texte = df_train.iloc[i]['message']  
    true_label = df_train.iloc[i]['label'] 
    
    mots = text2vect(texte, mots_inutiles)
    
    #calcule les proba de chaque label
    probas = {l: proba_exemple(mots, l, index_mots, frequences) for l in les_labels}
    
    #prédit le label avec la proba la + élevée
    predicted_label = max(probas, key=probas.get)
    
    #màj les cpt de bonnes ou mauvaises prédics
    if predicted_label == true_label:
        taux[true_label][True] += 1
    else:
        taux[true_label][False] += 1

# #############################################
toc = time.time()
print("Résultats des classements : ", taux)

print("Temps: ",toc-tic)
# le résultat total peut prendre un certain temps...


# Il reste à calculer les taux de bonne classification par label

Résultats des classements :  {'ham': {True: 1, False: 240}, 'spam': {True: 0, False: 37}}
Temps:  0.18259406089782715


<font color="RED" size="+1"><b>[Q]</b></font> Même question pour calculer le taux de bonne classification pour le dataset de test.

*Remarque*: L'index des mots ainsi que la matrice des fréquences restent les mêmes, ce sont toujours celles construites à partir de la base d'apprentissage. Par contre, la matrice des présences doit, elle, être recalculées.

In [171]:
# La variable suivante permettra de stocker les résultats
taux = dict()
toutes_probas = dict()
for l in les_labels:
    taux[l] = dict()
    taux[l][True] = 0    # nombre de bien classés
    taux[l][False] = 0   # nombre de mal classés
    toutes_probas[l] = np.zeros(1)
tic = time.time()

# ################################## A COMPLETER

mat_test = np.zeros((len(df_test), len(index_mots)), dtype=int)

for i in range(len(df_test)):
    texte = df_test.iloc[i]['message']  
    mots = text2vect(texte, mots_inutiles)  
    for mot in mots:
        if mot in index_mots:
            mat_test[i, index_mots.index(mot)] = 1 

for l in les_labels:
    log_proba = np.sum(mat_test * np.log(np.array([frequences[l].get(mot, 1e-10) for mot in index_mots])), axis=1)
    log_proba += np.sum((1 - mat_test) * np.log(1 - np.array([frequences[l].get(mot, 1e-10) for mot in index_mots])), axis=1)
    toutes_probas[l] = np.exp(log_proba)

for i in range(len(df_test)):
    p_ham = toutes_probas['ham'][i]
    p_spam = toutes_probas['spam'][i]
    
    predicted_label = 'spam' if p_spam > p_ham else 'ham'
    
    true_label = df_test.iloc[i]['label'] 
    
    if predicted_label == true_label:
        taux[true_label][True] += 1
    else:
        taux[true_label][False] += 1


# #############################################
toc = time.time()
print("Résultats des classements : ", taux)

print("Temps: ",toc-tic)
# le résultat total peut prendre un certain temps...


# Il reste à calculer les taux de bonne classification par label

C:\Users\inese\AppData\Local\Temp\ipykernel_20052\2966990756.py:23: RuntimeWarning: divide by zero encountered in log
  log_proba = np.sum(mat_test * np.log(np.array([frequences[l].get(mot, 1e-10) for mot in index_mots])), axis=1)
C:\Users\inese\AppData\Local\Temp\ipykernel_20052\2966990756.py:23: RuntimeWarning: invalid value encountered in multiply
  log_proba = np.sum(mat_test * np.log(np.array([frequences[l].get(mot, 1e-10) for mot in index_mots])), axis=1)
C:\Users\inese\AppData\Local\Temp\ipykernel_20052\2966990756.py:24: RuntimeWarning: divide by zero encountered in log
  log_proba += np.sum((1 - mat_test) * np.log(1 - np.array([frequences[l].get(mot, 1e-10) for mot in index_mots])), axis=1)
C:\Users\inese\AppData\Local\Temp\ipykernel_20052\2966990756.py:24: RuntimeWarning: invalid value encountered in log
  log_proba += np.sum((1 - mat_test) * np.log(1 - np.array([frequences[l].get(mot, 1e-10) for mot in index_mots])), axis=1)
C:\Users\inese\AppData\Local\Temp\ipykernel_20052\2

Résultats des classements :  {'ham': {True: 4584, False: 0}, 'spam': {True: 0, False: 710}}
Temps:  1.5947198867797852


## Evaluation du classifieur Naive Bayes

Pour tout ce que l'on a fait jusque-là, on a travaillé sur tout le dataset, pour pouvoir l'évaluer il est nécessaire de le séparer en ensemble d'apprentissage et ensemble de test.

<font color="RED" size="+1"><b>[Q]</b></font> Découper aléatoirement `df_spam` en 2 parties égales, chacune contenant des exemples des 2 labels, avec la même distribution des labels dans chaque partie. 
Une des parties sert à apprendre l'index des mots et leurs fréquences.
L'autre partie n'est utilisée que pour calculer le taux de bonne classification par label.

Donner ensuite les taux de bonne classification par label pour l'ensemble de train et pour l'ensemble de test.

*Remarque*: certains mots de la partie de test pourront ne pas être présents dans l'index de mots car ils peuvent être absents de la partie d'apprentissage.

<font color="RED" size="+1"><b>[Q]</b></font> Mettre en place une approche par validation croisée pour évaluer le taux de bonne classification moyen de cette approche. 

<font color="RED" size="+1"><b>[Q]</b></font> Comparer les résultats obtenus (taux de bonne classification avec la validation croisée, temps de calcul) avec ceux obtenus avec l'application d'un classifieur par $k$ plus proches voisins. Pour cela, la `mat_spam` doit être utilisée comme description des données. 

In [ ]:

def calcul_taux_bonne_classification(df, index_mots, frequences, les_labels):
    taux = {l: {True: 0, False: 0} for l in les_labels}
    
    for i in range(len(df)):
        texte = df.iloc[i]['message']  # Remplacez 'texte' par le nom correct de la colonne
        true_label = df.iloc[i]['label']
        mots = text2vect(texte, mots_inutiles)
        predicted_label = predire_label(mots, index_mots, frequences, les_labels)
        
        if predicted_label == true_label:
            taux[true_label][True] += 1
        else:
            taux[true_label][False] += 1
    
    for l in les_labels:
        total = taux[l][True] + taux[l][False]
        taux_bonne_classification = taux[l][True] / total if total > 0 else 0
        print(f"Taux de bonne classification pour {l}: {taux_bonne_classification:.2f}")
    
    return taux

def predire_label(exemple, index_mots, frequences, les_labels):
    probas = {l: proba_exemple(exemple, l, index_mots, frequences) for l in les_labels}
    return max(probas, key=probas.get)

# Données d'entrée
# df_spam : DataFrame contenant les textes et les labels
# mat_spam : matrice des occurrences des mots
# mots_inutiles : liste des mots à ignorer
# les_labels : liste des labels (ex: ['ham', 'spam'])

# Diviser le dataset en deux parties égales
df_train, df_test = train_test_split(df_spam, test_size=0.5, stratify=df_spam['label'], random_state=42)

# Construire l'index des mots et calculer les fréquences sur l'ensemble d'apprentissage
index_mots = set()
for texte in df_train['message']:  # Remplacez 'texte' par le nom correct de la colonne
    mots = text2vect(texte, mots_inutiles)
    index_mots.update(mots)
index_mots = list(index_mots)

frequences = {l: defaultdict(int) for l in les_labels}
for i in range(len(df_train)):
    texte = df_train.iloc[i]['message']  # Remplacez 'texte' par le nom correct de la colonne
    label = df_train.iloc[i]['label']
    mots = text2vect(texte, mots_inutiles)
    for mot in mots:
        frequences[label][mot] += 1

# Calculer les taux de bonne classification pour l'ensemble de train et de test
print("Résultats pour l'ensemble de train (Naive Bayes) :")
taux_train = calcul_taux_bonne_classification(df_train, index_mots, frequences, les_labels)

print("\nRésultats pour l'ensemble de test (Naive Bayes) :")
taux_test = calcul_taux_bonne_classification(df_test, index_mots, frequences, les_labels)

# Validation croisée pour Naive Bayes
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df_spam['message'])  # Remplacez 'texte' par le nom correct de la colonne
y = df_spam['label'].map({'ham': 1, 'spam': 0})

nb_classifier = MultinomialNB()
scores_nb = cross_val_score(nb_classifier, X, y, cv=5, scoring='accuracy')
print(f"\nTaux de bonne classification moyen (Naive Bayes, validation croisée) : {scores_nb.mean():.2f}")

# Comparaison avec k-NN
X = mat_spam  # Description des données
y = df_spam['label'].map({'ham': 1, 'spam': -1})  # Labels convertis en +1 (ham) et -1 (spam)

# Diviser les données en ensembles d'apprentissage et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)

# Initialiser le classifieur k-NN
knn_classifier = classif.ClassifierKNN(n_neighbors=5)

# Entraîner le classifieur k-NN
knn_classifier.fit(X_train, y_train)

# Évaluer le classifieur k-NN sur l'ensemble de test
y_pred = knn_classifier.predict(X_test)
taux_bonne_classification_knn = np.mean(y_pred == y_test)
print(f"\nTaux de bonne classification (k-NN) : {taux_bonne_classification_knn:.2f}")

# Validation croisée pour k-NN
scores_knn = cross_val_score(knn_classifier, X, y, cv=5, scoring='accuracy')
print(f"Taux de bonne classification moyen (k-NN, validation croisée) : {scores_knn.mean():.2f}")

# Comparaison des résultats
print("\nComparaison des résultats :")
print(f"- Naive Bayes (test) : {taux_test['ham'][True] / (taux_test['ham'][True] + taux_test['ham'][False]):.2f} pour ham, {taux_test['spam'][True] / (taux_test['spam'][True] + taux_test['spam'][False]):.2f} pour spam")
print(f"- k-NN (test) : {taux_bonne_classification_knn:.2f}")


Résultats pour l'ensemble de train (Naive Bayes) :
Taux de bonne classification pour ham: 0.05
Taux de bonne classification pour spam: 0.02

Résultats pour l'ensemble de test (Naive Bayes) :
Taux de bonne classification pour ham: 0.07
Taux de bonne classification pour spam: 0.24

Taux de bonne classification moyen (Naive Bayes, validation croisée) : 0.98


NameError: name 'mat_spam' is not defined